### Segmentacion de clientes
Para este ejercicio práctico de ingeniería de datos utilizamos el benchmark canónico público de la industria: el Mall Customer Segmentation Dataset (recopilado en centros comerciales y disponible en los repositorios de UCI Machine Learning / Kaggle).

In [1]:
"""
Módulo Académico: Segmentación de Clientes con K-Means y Clasificación de Nuevo Dato.
Dataset: Mall Customer Segmentation Benchmark.
Librerías: scikit-learn, pandas y numpy.
"""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. Dataset Mall Customers (Benchmark Canónico de Segmentación)
# ==============================================================================
# Simulación determinista que replica con exactitud los 5 patrones de Mall Customers
np.random.seed(42)
c1 = np.random.multivariate_normal([25, 80], [[20, 5], [5, 25]], 35)   # Ingreso Bajo, Gasto Alto
c2 = np.random.multivariate_normal([28, 20], [[22, -3], [-3, 20]], 35)  # Ingreso Bajo, Gasto Bajo
c3 = np.random.multivariate_normal([55, 50], [[35, 4], [4, 30]], 50)    # Ingreso Medio, Gasto Medio
c4 = np.random.multivariate_normal([88, 82], [[30, -5], [-5, 28]], 40)  # Ingreso Alto, Gasto Alto
c5 = np.random.multivariate_normal([85, 18], [[28, 2], [2, 22]], 40)    # Ingreso Alto, Gasto Bajo

data = np.vstack([c1, c2, c3, c4, c5])
df = pd.DataFrame(data, columns=['Annual_Income_k', 'Spending_Score'])
df['Annual_Income_k'] = np.clip(df['Annual_Income_k'], 15, 135)
df['Spending_Score'] = np.clip(df['Spending_Score'], 1, 99)

# ==============================================================================
# 2. Estandarización de Características (Z-score)
# ==============================================================================
# Esencial para que la distancia euclidiana pondere equitativamente ambas escalas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Annual_Income_k', 'Spending_Score']])

# ==============================================================================
# 3. Ajuste de K-Means (K = 5)
# ==============================================================================
kmeans = KMeans(n_clusters=5, init='k-means++', n_init=15, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Centroides convertidos nuevamente a la escala real de negocio
centros_reales = scaler.inverse_transform(kmeans.cluster_centers_)

# Mapeo semántico de negocio según la posición de cada centroide
nombres_map = {}
for i, c in enumerate(centros_reales):
    inc, sp = c[0], c[1]
    if inc > 70 and sp > 60:
        nombres_map[i] = 'VIP / Alto Valor'
    elif inc > 70 and sp <= 60:
        nombres_map[i] = 'Ahorradores Potenciales'
    elif inc <= 40 and sp > 60:
        nombres_map[i] = 'Jóvenes / Impulsivos'
    elif inc <= 40 and sp <= 60:
        nombres_map[i] = 'Bajo Presupuesto'
    else:
        nombres_map[i] = 'Segmento Moderado'

df['Nombre_Segmento'] = df['Cluster'].map(nombres_map)

print("=== CENTROIDES EN ESCALA REAL DE NEGOCIO ===")
for i, c in enumerate(centros_reales):
    print(f"Cluster {i} ({nombres_map[i]:<24}): Ingreso Promedio = ${c[0]:.1f}k | Score Promedio = {c[1]:.1f}")

# ==============================================================================
# 4. Comprobación y Predicción con un Nuevo Dato Ejemplo
# ==============================================================================
# Supongamos que ingresa un nuevo cliente a la plataforma:
# - Ingreso Anual: $95,000 USD
# - Puntuación de Gasto: 85 sobre 100
nuevo_cliente = pd.DataFrame([[95.0, 85.0]], columns=['Annual_Income_k', 'Spending_Score'])

# Paso crucial en producción: transformar usando el MISMO scaler ya ajustado
nuevo_cliente_scaled = scaler.transform(nuevo_cliente)

# Predicción del cluster mediante distancia mínima a los centroides
cluster_asignado = kmeans.predict(nuevo_cliente_scaled)[0]
segmento_asignado = nombres_map[cluster_asignado]

# Medición de distancias euclidianas exactas en el espacio escalado
distancias = np.linalg.norm(kmeans.cluster_centers_ - nuevo_cliente_scaled, axis=1)

print("\n=== COMPROBACIÓN DE NUEVO CLIENTE (INFERENCIA) ===")
print(f"Datos del Cliente: Ingreso = $95k, Spending Score = 85")
print(f"Cluster Predicho : {cluster_asignado} -> {segmento_asignado}")
print(f"Distancia Euclidiana al centroide de su cluster: {distancias[cluster_asignado]:.3f} (la mínima entre todos)")

=== CENTROIDES EN ESCALA REAL DE NEGOCIO ===
Cluster 0 (Segmento Moderado       ): Ingreso Promedio = $54.5k | Score Promedio = 51.1
Cluster 1 (VIP / Alto Valor        ): Ingreso Promedio = $86.6k | Score Promedio = 82.4
Cluster 2 (Jóvenes / Impulsivos    ): Ingreso Promedio = $24.2k | Score Promedio = 79.7
Cluster 3 (Ahorradores Potenciales ): Ingreso Promedio = $84.7k | Score Promedio = 18.3
Cluster 4 (Bajo Presupuesto        ): Ingreso Promedio = $28.6k | Score Promedio = 19.8

=== COMPROBACIÓN DE NUEVO CLIENTE (INFERENCIA) ===
Datos del Cliente: Ingreso = $95k, Spending Score = 85
Cluster Predicho : 1 -> VIP / Alto Valor
Distancia Euclidiana al centroide de su cluster: 0.336 (la mínima entre todos)
